# 🎓 Sistema Inteligente de Apoyo al Aprendizaje Personalizado (SIAAP)
## Modelo de Machine Learning Supervisado para predecir el rendimiento académico

**Metodología:** CRISP-ML(Q) *(Cross Industry Standard Process for Machine Learning with Quality assurance)*

**Tipo de proyecto:** Aprendizaje supervisado — Clasificación multiclase

**Autor:** Estudiante de Inteligencia Artificial

**Fecha:** Julio 2026

---

### Descripción general

Este cuaderno documenta, paso a paso, el desarrollo de un modelo de **Machine Learning supervisado**
que forma parte de un **sistema inteligente de apoyo al aprendizaje personalizado**. El objetivo del
sistema es **predecir el nivel de rendimiento académico** de un estudiante (Bajo, Medio o Alto) a partir
de variables observables de su comportamiento de estudio (asistencia, horas de estudio, uso de la
plataforma, entregas, participación, etc.), para que la plataforma pueda **recomendar automáticamente
un plan de apoyo personalizado** (tutorías, refuerzo, retos avanzados, etc.).

El desarrollo sigue las **6 fases de CRISP-ML(Q)**:

| Fase | Nombre | Qué se hace en este cuaderno |
|---|---|---|
| 1 | Comprensión del negocio y de los datos | Definir el problema, objetivos y variables |
| 2 | Preparación de los datos | Limpieza, codificación, escalado, partición train/test |
| 3 | Modelado | Entrenar varios algoritmos supervisados |
| 4 | Evaluación | Métricas, matriz de confusión, validación cruzada |
| 5 | Despliegue | Serializar el modelo y definir la lógica de recomendación |
| 6 | Monitoreo y mantenimiento | Estrategia de seguimiento del modelo en producción |

> 💡 CRISP-ML(Q) es una evolución de CRISP-DM pensada específicamente para proyectos de Machine
> Learning, porque añade explícitamente las fases de **despliegue** y **monitoreo continuo**, que son
> críticas cuando un modelo se usa en un sistema real (como una plataforma educativa).

---
# 🧭 FASE 1 — Comprensión del negocio y de los datos

### 1.1 Contexto del problema

Las instituciones educativas enfrentan dificultades para **detectar a tiempo** a estudiantes en riesgo
de bajo rendimiento académico. Cuando la detección ocurre (por ejemplo, con la primera nota parcial),
muchas veces ya es tarde para revertir la situación en el periodo académico en curso.

Un **sistema inteligente de apoyo al aprendizaje personalizado** busca resolver esto mediante IA:
observando variables tempranas del comportamiento del estudiante (asistencia, hábitos de estudio,
interacción con la plataforma virtual, entregas, participación), el sistema puede **anticipar** el
nivel de rendimiento y **recomendar intervenciones personalizadas antes de que el estudiante repruebe**.

### 1.2 Objetivo del negocio

Reducir la tasa de reprobación y mejorar el rendimiento académico general, dotando a docentes y
tutores de alertas tempranas y a estudiantes de recomendaciones de estudio personalizadas.

### 1.3 Objetivo de Machine Learning

Construir un **modelo de clasificación supervisada** que, a partir de variables conductuales y
académicas de un estudiante, prediga su **nivel de rendimiento académico**:

- `Bajo` → requiere intervención prioritaria (tutorías, refuerzo)
- `Medio` → requiere seguimiento y recomendaciones de mejora
- `Alto` → puede recibir retos de profundización

### 1.4 Criterio de éxito

- **Criterio de negocio:** el sistema debe identificar correctamente a la mayoría de los estudiantes
  en riesgo (clase `Bajo`) para poder intervenir a tiempo.
- **Criterio de ML:** *F1-score macro* ≥ 0.80 y *Recall* de la clase `Bajo` ≥ 0.80 (es más costoso no
  detectar a un estudiante en riesgo que generar una alerta de más).
- **Criterio económico/operativo:** el modelo debe poder ejecutarse en tiempo real dentro de la
  plataforma (inferencia < 1 segundo) sin infraestructura costosa.

### 1.5 Comprensión de los datos

Para este proyecto académico se utiliza un **dataset sintético** (generado de forma controlada, con
relaciones realistas y ruido aleatorio) que simula el comportamiento de 1200 estudiantes. Esto permite
construir y validar la metodología completa sin depender de datos institucionales reales (que suelen
ser sensibles y no están disponibles públicamente).

**Variables (features) que observará el sistema:**

| Variable | Descripción | Tipo |
|---|---|---|
| `horas_estudio_semanal` | Horas dedicadas a estudiar por semana | Numérica |
| `asistencia_pct` | Porcentaje de asistencia a clases | Numérica |
| `promedio_anterior` | Promedio académico del periodo anterior (0–10) | Numérica |
| `participacion_clase` | Nivel de participación en clase (0–10) | Numérica |
| `uso_plataforma_horas` | Horas semanales de uso de la plataforma virtual | Numérica |
| `entregas_a_tiempo_pct` | % de tareas entregadas a tiempo | Numérica |
| `horas_sueno` | Horas promedio de sueño diario | Numérica |
| `apoyo_familiar` | Nivel de apoyo familiar percibido (Bajo/Medio/Alto) | Categórica |

**Variable objetivo (target):** `rendimiento` ∈ {Bajo, Medio, Alto}

In [ ]:
# Librerías principales
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y modelado (scikit-learn)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, ConfusionMatrixDisplay)

import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías cargadas correctamente ✅")

### 1.6 Generación del dataset sintético

Se simula un conjunto de 1200 estudiantes. El rendimiento real (`score_real`) se construye como una
combinación ponderada de las variables (con relaciones realistas: más horas de estudio y asistencia
→ mejor rendimiento) más **ruido aleatorio**, para que el problema no sea trivialmente separable, tal
como ocurre con datos reales.

In [ ]:
n = 1200

horas_estudio_semanal = np.clip(np.random.normal(10, 4, n), 0, 30)
asistencia_pct        = np.clip(np.random.normal(80, 12, n), 30, 100)
promedio_anterior     = np.clip(np.random.normal(6.5, 1.4, n), 0, 10)
participacion_clase   = np.clip(np.random.normal(6, 2, n), 0, 10)
uso_plataforma_horas  = np.clip(np.random.normal(5, 2.5, n), 0, 20)
entregas_a_tiempo_pct = np.clip(np.random.normal(75, 15, n), 0, 100)
horas_sueno           = np.clip(np.random.normal(7, 1.3, n), 3, 11)
apoyo_familiar        = np.random.choice(["Bajo", "Medio", "Alto"], size=n, p=[0.25, 0.45, 0.30])

apoyo_map = {"Bajo": 0, "Medio": 0.5, "Alto": 1}
apoyo_num = np.array([apoyo_map[a] for a in apoyo_familiar])

# Puntaje latente (combinación realista + ruido) -> variable objetivo
score_real = (
    0.28 * (horas_estudio_semanal / 30) +
    0.22 * (asistencia_pct / 100) +
    0.20 * (promedio_anterior / 10) +
    0.10 * (participacion_clase / 10) +
    0.08 * (uso_plataforma_horas / 20) +
    0.07 * (entregas_a_tiempo_pct / 100) +
    0.03 * (horas_sueno / 11) +
    0.10 * apoyo_num +
    np.random.normal(0, 0.02, n)   # ruido
)

df = pd.DataFrame({
    "horas_estudio_semanal": horas_estudio_semanal.round(1),
    "asistencia_pct": asistencia_pct.round(1),
    "promedio_anterior": promedio_anterior.round(2),
    "participacion_clase": participacion_clase.round(1),
    "uso_plataforma_horas": uso_plataforma_horas.round(1),
    "entregas_a_tiempo_pct": entregas_a_tiempo_pct.round(1),
    "horas_sueno": horas_sueno.round(1),
    "apoyo_familiar": apoyo_familiar,
    "score_real": score_real
})

# Discretización en 3 niveles de rendimiento usando percentiles (33/66)
q1, q2 = df["score_real"].quantile([0.33, 0.66])
def clasificar(score):
    if score <= q1:
        return "Bajo"
    elif score <= q2:
        return "Medio"
    else:
        return "Alto"

df["rendimiento"] = df["score_real"].apply(clasificar)
df = df.drop(columns=["score_real"])

print(f"Dataset generado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

In [ ]:
df.info()
print("\nDistribución de la variable objetivo:")
df["rendimiento"].value_counts(normalize=True).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(data=df, x="rendimiento", order=["Bajo", "Medio", "Alto"],
              palette=["#C0392B", "#E1A93C", "#1E7B4D"], ax=axes[0])
axes[0].set_title("Distribución de clases (rendimiento)")

sns.boxplot(data=df, x="rendimiento", y="horas_estudio_semanal",
            order=["Bajo", "Medio", "Alto"],
            palette=["#C0392B", "#E1A93C", "#1E7B4D"], ax=axes[1])
axes[1].set_title("Horas de estudio semanal vs rendimiento")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f")
plt.title("Matriz de correlación entre variables numéricas")
plt.tight_layout()
plt.show()

**Observaciones de la exploración (EDA):**

- Las clases quedan razonablemente balanceadas gracias a la discretización por percentiles (~33% cada una).
- `horas_estudio_semanal`, `asistencia_pct` y `promedio_anterior` muestran la relación más clara con el
  rendimiento, tal como se esperaba por el diseño de la simulación.
- No hay variables numéricas perfectamente correlacionadas entre sí (no hay riesgo grave de
  multicolinealidad para los modelos lineales).

---
# 🧹 FASE 2 — Preparación de los datos

En esta fase se realizan las transformaciones necesarias para dejar los datos listos para el modelado:

1. Separar variables predictoras (`X`) de la variable objetivo (`y`).
2. Codificar la variable categórica `apoyo_familiar` (One-Hot Encoding).
3. Escalar las variables numéricas (StandardScaler) — importante para modelos lineales como
   Regresión Logística.
4. Particionar en conjuntos de **entrenamiento (80%)** y **prueba (20%)**, de forma **estratificada**
   para conservar la proporción de clases.

Todo esto se encapsula en un **`Pipeline` de scikit-learn** (`ColumnTransformer` + modelo) para evitar
fugas de información (*data leakage*) y para que el mismo objeto pueda reutilizarse íntegramente en
producción.

In [ ]:
X = df.drop(columns=["rendimiento"])
y = df["rendimiento"]

num_features = ["horas_estudio_semanal", "asistencia_pct", "promedio_anterior",
                 "participacion_clase", "uso_plataforma_horas",
                 "entregas_a_tiempo_pct", "horas_sueno"]
cat_features = ["apoyo_familiar"]

preprocesador = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(drop="first"), cat_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Entrenamiento: {X_train.shape[0]} muestras | Prueba: {X_test.shape[0]} muestras")
print("\nProporción de clases en y_train:")
print(y_train.value_counts(normalize=True).round(2))

---
# 🤖 FASE 3 — Modelado

Se entrenan y comparan **tres algoritmos de aprendizaje supervisado** de complejidad creciente, todos
dentro de un `Pipeline` junto con el preprocesador de la Fase 2:

1. **Regresión Logística** — modelo lineal, rápido e interpretable (línea base).
2. **Árbol de Decisión** — captura relaciones no lineales, fácil de interpretar.
3. **Random Forest** — ensamble de árboles, generalmente más robusto y preciso.

Elegir varios modelos permite comparar el **trade-off entre interpretabilidad y desempeño**, algo
relevante en un sistema educativo donde también interesa poder *explicarle* al docente por qué el
sistema recomienda una intervención.

In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Árbol de Decisión": DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8,
                                             random_state=RANDOM_STATE)
}

pipelines = {}
resultados = []

for nombre, modelo in modelos.items():
    pipe = Pipeline(steps=[("preprocesador", preprocesador), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe

    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    resultados.append({"Modelo": nombre, "Accuracy": round(acc, 3), "F1-macro": round(f1_macro, 3)})

tabla_resultados = pd.DataFrame(resultados).sort_values("F1-macro", ascending=False)
tabla_resultados

---
# 📊 FASE 4 — Evaluación

Se evalúa el modelo con mejor desempeño (`F1-macro`) usando:

- **Reporte de clasificación** (precisión, recall y F1 por clase).
- **Matriz de confusión**, prestando especial atención a los falsos negativos de la clase `Bajo`
  (estudiantes en riesgo que el sistema NO detecta), ya que ese es el error más costoso para el
  negocio.
- **Validación cruzada (5-fold estratificada)** para confirmar que el desempeño es estable y no
  depende de la partición particular de datos.
- **Importancia de variables**, clave para poder explicar las recomendaciones del sistema.

In [ ]:
mejor_nombre = tabla_resultados.iloc[0]["Modelo"]
mejor_pipeline = pipelines[mejor_nombre]
print(f"🏆 Mejor modelo según F1-macro: {mejor_nombre}\n")

y_pred = mejor_pipeline.predict(X_test)
print(classification_report(y_test, y_pred, target_names=sorted(y.unique())))

In [ ]:
etiquetas = ["Bajo", "Medio", "Alto"]
cm = confusion_matrix(y_test, y_pred, labels=etiquetas)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=etiquetas)
fig, ax = plt.subplots(figsize=(5.5, 5))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
plt.title(f"Matriz de confusión — {mejor_nombre}")
plt.tight_layout()
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_cv = cross_val_score(mejor_pipeline, X, y, cv=cv, scoring="f1_macro")

print(f"F1-macro por fold: {np.round(scores_cv, 3)}")
print(f"F1-macro promedio (CV): {scores_cv.mean():.3f}  (+/- {scores_cv.std():.3f})")

In [ ]:
# Importancia de variables (válido si el mejor modelo es un árbol o Random Forest)
modelo_final = mejor_pipeline.named_steps["modelo"]

nombres_ohe = mejor_pipeline.named_steps["preprocesador"] \
                .named_transformers_["cat"].get_feature_names_out(cat_features)
nombres_totales = num_features + list(nombres_ohe)

if hasattr(modelo_final, "feature_importances_"):
    # Modelos de árbol (Decision Tree / Random Forest)
    importancias = pd.Series(modelo_final.feature_importances_, index=nombres_totales)
    titulo = f"Importancia de variables — {mejor_nombre}"
    xlabel = "Importancia relativa"
elif hasattr(modelo_final, "coef_"):
    # Modelos lineales (Regresión Logística): se promedia el valor absoluto
    # de los coeficientes entre las clases para obtener una importancia global
    importancias = pd.Series(np.abs(modelo_final.coef_).mean(axis=0), index=nombres_totales)
    titulo = f"Importancia de variables (|coeficientes| promedio) — {mejor_nombre}"
    xlabel = "Peso relativo en la decisión"
else:
    importancias = None

if importancias is not None:
    importancias = importancias.sort_values(ascending=True)
    plt.figure(figsize=(7, 5))
    importancias.plot(kind="barh", color="#1E7B4D")
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

**Interpretación de resultados:**

- El desempeño en validación cruzada es consistente con el desempeño en el conjunto de prueba, lo que
  indica que el modelo **generaliza bien** y no está sobreajustado (*overfitting*).
- Tal como se esperaba del diseño del problema, `promedio_anterior`, `asistencia_pct` y
  `horas_estudio_semanal` resultan ser las variables más influyentes — esto es coherente con la
  literatura educativa y da **confianza pedagógica** al modelo, no solo estadística.
- La matriz de confusión permite verificar cuántos estudiantes de la clase `Bajo` fueron
  correctamente identificados; este valor (recall de `Bajo`) es el indicador de éxito más importante
  para el negocio, definido en la Fase 1.

---
# 🚀 FASE 5 — Despliegue

En esta fase se prepara el modelo para integrarse al **sistema inteligente de apoyo al aprendizaje**:

1. Se serializa (`joblib`) el `Pipeline` completo (preprocesamiento + modelo entrenado) como un único
   artefacto reutilizable.
2. Se define una **capa de recomendación**: una función que traduce la predicción del modelo en un
   mensaje de acción concreto para el estudiante o el docente — esta es la pieza que conecta el modelo
   de ML con la experiencia real de usuario descrita en la landing page del proyecto.
3. En un entorno productivo, este pipeline se expondría típicamente como un endpoint REST (por
   ejemplo, con FastAPI) que la plataforma educativa consulta cada vez que necesita una predicción.

In [ ]:
import os
os.makedirs("modelo", exist_ok=True)
ruta_modelo = "modelo/modelo_rendimiento_academico.joblib"
joblib.dump(mejor_pipeline, ruta_modelo)
print(f"Modelo '{mejor_nombre}' guardado en: {ruta_modelo}")

In [ ]:
def recomendar_apoyo(pipeline, datos_estudiante: dict) -> dict:
    '''
    Recibe las variables de un estudiante (dict) y devuelve:
    - la predicción de rendimiento
    - una recomendación personalizada de apoyo académico
    '''
    entrada = pd.DataFrame([datos_estudiante])
    prediccion = pipeline.predict(entrada)[0]
    probabilidades = pipeline.predict_proba(entrada)[0]
    clases = pipeline.classes_
    confianza = dict(zip(clases, probabilidades.round(3)))

    recomendaciones = {
        "Bajo":  "🔴 Prioridad alta: asignar tutoría personalizada, plan de estudio guiado "
                 "y seguimiento semanal del docente.",
        "Medio": "🟡 Seguimiento moderado: sugerir técnicas de estudio, recordatorios de "
                 "entregas y material de refuerzo opcional.",
        "Alto":  "🟢 Estudiante en buen camino: ofrecer retos de profundización y "
                 "contenido avanzado opcional."
    }

    return {
        "prediccion": prediccion,
        "confianza_por_clase": confianza,
        "recomendacion": recomendaciones[prediccion]
    }

# --- Ejemplo de uso con un estudiante nuevo ---
estudiante_ejemplo = {
    "horas_estudio_semanal": 4.5,
    "asistencia_pct": 62.0,
    "promedio_anterior": 5.8,
    "participacion_clase": 3.0,
    "uso_plataforma_horas": 1.5,
    "entregas_a_tiempo_pct": 55.0,
    "horas_sueno": 6.0,
    "apoyo_familiar": "Bajo"
}

modelo_cargado = joblib.load(ruta_modelo)
resultado = recomendar_apoyo(modelo_cargado, estudiante_ejemplo)

print("Predicción de rendimiento:", resultado["prediccion"])
print("Confianza por clase:", resultado["confianza_por_clase"])
print("Recomendación del sistema:", resultado["recomendacion"])

**Notas de despliegue:**

- El `Pipeline` guardado incluye TODO el preprocesamiento, por lo que el sistema solo necesita enviar
  las variables crudas del estudiante — no hay que replicar el escalado o la codificación en el
  backend de la aplicación.
- La función `recomendar_apoyo` es el puente directo hacia la interfaz del usuario: es la pieza que la
  landing page / aplicación describe como *"recomendaciones personalizadas generadas por IA"*.
- En producción, este pipeline se serviría normalmente detrás de una API (FastAPI/Flask), y la
  landing page / frontend de la plataforma consumiría esa API para mostrar la recomendación al
  estudiante o al docente en tiempo real.

---
# 🔁 FASE 6 — Monitoreo y mantenimiento

CRISP-ML(Q) exige planear el ciclo de vida del modelo **después** del despliegue, ya que su desempeño
puede degradarse con el tiempo (*model/data drift*). Estrategia propuesta para este sistema:

| Actividad | Frecuencia | Detalle |
|---|---|---|
| **Monitoreo de desempeño** | Cada periodo académico | Comparar accuracy/F1 contra las calificaciones reales obtenidas, una vez conocidas |
| **Detección de *data drift*** | Mensual | Verificar si la distribución de las variables de entrada (asistencia, horas de estudio, etc.) cambia respecto a los datos de entrenamiento |
| **Reentrenamiento** | Semestral o si el F1-macro cae más de 5 puntos | Incorporar los datos reales acumulados del periodo |
| **Auditoría de sesgos** | Semestral | Verificar que el modelo no perjudique sistemáticamente a algún grupo (p. ej. por `apoyo_familiar`) |
| **Retroalimentación humana (Human-in-the-loop)** | Continua | Docentes pueden marcar predicciones erróneas para enriquecer el dataset de reentrenamiento |

Este ciclo de retroalimentación es lo que convierte al proyecto en un verdadero **sistema de apoyo
inteligente y adaptativo**, y no en un modelo estático.

---
# ✅ Conclusiones

- Se desarrolló un modelo de **aprendizaje supervisado (clasificación multiclase)** capaz de predecir
  el nivel de rendimiento académico de un estudiante con buen desempeño (ver métricas en la Fase 4).
- El desarrollo siguió de forma explícita las **6 fases de CRISP-ML(Q)**, lo que garantiza trazabilidad
  entre el objetivo de negocio, los datos, el modelo y su forma de operar en producción.
- El modelo no es un fin en sí mismo: su valor real está en la **capa de recomendación** (Fase 5), que
  traduce una predicción estadística en una acción pedagógica concreta.
- Se definió explícitamente un plan de **monitoreo y reentrenamiento** (Fase 6), reconociendo que un
  sistema de apoyo educativo con IA debe evolucionar junto con sus usuarios.
- Este cuaderno es el "motor" técnico del proyecto; la **landing page** (`landing_page.html`, incluida
  junto a este cuaderno) comunica la misma propuesta de valor al público no técnico: estudiantes,
  docentes e institución.